# AgentIR v3: meeting action items on 200 random dialogs (run all)

**Runtime → Change runtime type → L4 (or T4) GPU**, then **Runtime → Run all**. Results go to `MyDrive/agentir-experiments/runs/v3_*`; rerunning resumes every step.

Fixed sample: `configs/subsets/random200_seed20261112.json` (200 of the 2,064 VisDial val dialogs, seed 20261112), full 10 rounds, Qwen3.5-9B.

| Step | What | Ollama on L4 | OpenRouter API |
|---|---|---|---|
| 0–2 | setup, BLIP index (reused from Drive), 200 target images | ~10 min | ~10 min |
| 3 | **Synthetic upper bound**: target caption forced into the 10 shown | ~4 h | ~20–40 min, ~\$2–3 |
| 4 | Vanilla PlugIR (reference) | ~4 h | ~20–40 min, ~\$2–3 |
| 5 | Cluster diagnostics + Hits@10 curves (no LLM) | ~5 min | ~5 min |
| 6 | History pooling + its matched baseline (fixed top-100) | ~8 h | ~1 h, ~\$5 |
| 7 | Re-runs: fixed 500 pool, rank-aware representatives | ~8 h | ~1 h, ~\$5 |

API prices: Qwen3.5-9B on OpenRouter, \$0.10/M input and \$0.15/M output tokens (Nov 2026). API and Ollama runs get different fingerprints (full precision vs 4-bit), so compare methods only within one backend.


## 0. Choose the backend

In [ ]:
# 'ollama' = free, Qwen3.5-9B 4-bit on this GPU (slow: ~4 h per method for 200 dialogs)
# 'openrouter' = paid API, same model, ~10x faster. Add your key under Colab's 🔑 Secrets
#                as OPENROUTER_API_KEY (Notebook access on). Never paste it into a cell.
BACKEND = 'ollama'
SUBSET = 'configs/subsets/random200_seed20261112.json'
TAG = 'v3' if BACKEND == 'ollama' else 'v3api'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf ChatIR-Cornell2026 && git clone -q -b experiments-matan https://github.com/nopharshalom/ChatIR-Cornell2026
%cd /content/ChatIR-Cornell2026
!git log --oneline -1
!pip install -q -e ".[dev]" spacy && python -m spacy download en_core_web_sm > /tmp/spacy.log 2>&1; tail -1 /tmp/spacy.log
!pytest -q | tail -1

In [ ]:
import os, time, json, glob
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/agentir-experiments'
!mkdir -p "{SAVE_DIR}/cache" "{SAVE_DIR}/runs"
!rm -rf cache runs && ln -s "{SAVE_DIR}/cache" cache && ln -s "{SAVE_DIR}/runs" runs
B = '-c configs/base.yaml -c configs/backends/colab.yaml'
if BACKEND == 'openrouter':
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    B += ' -c configs/backends/openrouter.yaml'
B += f' --set data.subset_file={SUBSET} --set agent.llm.cache_path=/content/llm_cache_{TAG}.sqlite'
if os.path.exists(f'{SAVE_DIR}/cache/llm_cache_{TAG}.sqlite') and not os.path.exists(f'/content/llm_cache_{TAG}.sqlite'):
    !cp "{SAVE_DIR}/cache/llm_cache_{TAG}.sqlite" /content/   # resume: reuse earlier LLM calls
def tick(name, t0):
    m = round((time.time() - t0) / 60, 1); print(f'[{name}] {m} min')
    open(f'{SAVE_DIR}/runs/timings.txt', 'a').write(f'{TAG}:{name}\t{m} min\n')
    !cp /content/llm_cache_{TAG}.sqlite "{SAVE_DIR}/cache/" 2>/dev/null || true
def run(exp):
    t0 = time.time()
    !agentir run {B} -c configs/experiments/{exp}.yaml --set name={TAG}_{exp} | tail -8
    tick(exp, t0)
def run_dir(exp):
    d = [d for d in glob.glob(f'runs/{TAG}_{exp}-*') if os.path.exists(d + '/sessions.jsonl')]
    return max(d, key=os.path.getmtime) if d else None
print(B)

## 1. BLIP index (reused from Drive) and the 200 target images
Only the targets are needed (the simulated user looks at them); search runs on the cached index.

In [ ]:
t0 = time.time()
assert os.path.exists('cache/index_blip.pt'), 'index missing from Drive: run AgentIR_Experiments_RunAll_v2 section 2 once'
import urllib.request
from concurrent.futures import ThreadPoolExecutor
Q = json.load(open('dialogues/VisDial_v1.0_queries_val.json'))
need = [Q[i]['img'] for i in json.load(open(SUBSET))]
def fetch(rel):
    dest = f'/content/data/{rel}'
    if not os.path.exists(dest):
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        urllib.request.urlretrieve('http://images.cocodataset.org/' + rel, dest)
with ThreadPoolExecutor(16) as ex: list(ex.map(fetch, need))
print(sum(os.path.exists(f'/content/data/{r}') for r in need), 'of', len(need), 'target images present')
tick('images', t0)

## 2. Qwen3.5-9B (Ollama only; skipped for the API)

In [ ]:
if BACKEND == 'ollama':
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -2 /tmp/ollama_install.log
    import subprocess, urllib.request
    env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_KEEP_ALIVE='-1')
    ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try: urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2); print('ollama is up'); break
        except Exception: time.sleep(1)
    !ollama pull qwen3.5:9b > /tmp/pull.log 2>&1; tail -1 /tmp/pull.log
img = '/content/data/' + need[0]
!agentir check-llm {B} --set agent.llm.cache_path=null --image {img}

## 3. Synthetic upper bound: target caption forced into the 10 shown captions
Cheats on purpose: tells us how much better PlugIR gets if the candidate evidence always contains the target.

In [ ]:
run('oracle_target')

## 4. Vanilla PlugIR on the same 200 dialogs (the reference curve)

In [ ]:
run('plugir')

## 5. Diagnostics and curves (no LLM, a few minutes)
Cluster/target diagnostics on vanilla PlugIR (includes the "target is ever a representative" rate), then Hits@10 / Recall@10 curves for everything finished so far. Rerun this cell any time.

In [ ]:
def report():
    t0 = time.time()
    if run_dir('plugir'):
        !python experiments/e5_cluster_diagnostics.py {B} --run {run_dir('plugir')} --out runs/{TAG}_e5_plugir | tail -45
    labels = [('PlugIR (vanilla)', 'plugir'), ('Target caption forced in (upper bound)', 'oracle_target'),
              ('History pool, top-100', 'cumulative_pool'), ('Fixed pool, top-100 (matched)', 'fixed_pool_100'),
              ('Fixed pool, top-500', 'e1_fixed_pool'), ('Rank-aware representatives', 'e1_rep_rank')]
    specs = ' '.join(f'"{l}={run_dir(e)}"' for l, e in labels if run_dir(e))
    if run_dir('plugir'):
        !python experiments/plot_curves.py --out runs/{TAG}_curves {specs}
    tick('report', t0)
report()
from IPython.display import Image, display
for p in [f'runs/{TAG}_curves/hits10_curves.png', f'runs/{TAG}_curves/recall10_curves.png'] + sorted(glob.glob(f'runs/{TAG}_e5_plugir/figs/*.png'))[:2]:
    if os.path.exists(p): display(Image(p, width=760))

## 6. History pooling and its matched baseline
Union of the top-200 images of every round's query, re-ranked by the current query, top 100 clustered; versus clustering the current query's top 100.

In [ ]:
run('cumulative_pool')
run('fixed_pool_100')

## 7. Earlier variants re-run on the 200 dialogs

In [ ]:
run('e1_fixed_pool')
run('e1_rep_rank')

## 8. Final report

In [ ]:
report()
for p in sorted(glob.glob(f'runs/{TAG}_curves/*.png')):
    display(Image(p, width=760))
print(open(f'runs/{TAG}_curves/curves.json').read()[-2500:])